# Diagnostic Tests II: Functional Form, Normality & Outliers. Lecture Notebook
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C., *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 4-5

---
**Learning Objectives:**
- **Test** the functional form with **Ramsey RESET** (powers of the fitted values, F-test)
- **Repair** a misspecified form with a **log transformation**, and test again
- **Test** residual normality with **Jarque-Bera** (skewness and kurtosis)
- **Detect** outliers with **standardized residuals** and handle them with **event dummies**

> Run each cell with **Shift+Enter**. This notebook accompanies the V9 lecture slides.
> The slides use illustrative values; the code below computes the live numbers from current data. They will be close, but not identical.

## Step 0: Install & Import Libraries

In [ ]:
!pip install yfinance pandas-datareader statsmodels --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset
from statsmodels.stats.stattools import jarque_bera
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Part 1: The Two Running Examples

1. **Bond price on yield** (weekly levels), our functional-form patient. We price a hypothetical 10-year zero-coupon bond off the 10-year Treasury yield: $P_t = 100/(1 + y_t/100)^{10}$, plus a small amount of pricing noise (think bid-ask bounce and stale quotes; it also keeps the example statistically honest). Bond pricing is CONVEX, so a linear regression of $P$ on $y$ is misspecified by construction. A perfect laboratory.
2. **Apple CAPM residuals** (daily returns), our normality and outlier patient.

### 1.1 Bond price on yield (weekly levels)

In [ ]:
y10 = web.DataReader('DGS10', 'fred', '2010-01-01', '2024-12-31')
y_w = y10.resample('W-THU').mean().dropna()
y_w.columns = ['y10']

bond = pd.DataFrame({'y10': y_w['y10']})
rng  = np.random.default_rng(42)                # fixed seed: everyone gets the same numbers
lnP  = np.log(100) - 10*np.log(1 + bond['y10']/100) + rng.normal(0, 0.004, len(bond))
bond['P'] = np.exp(lnP)                          # 10y zero-coupon price + small pricing noise

X_b   = sm.add_constant(bond['y10'])
m_lvl = sm.OLS(bond['P'], X_b).fit()

print(f'n = {len(bond)} weeks')
print(f'Linear levels model: slope = {m_lvl.params["y10"]:.3f}, R² = {m_lvl.rsquared:.4f}')
print('A high R² and STILL misspecified, as the next steps show. R² is not a specification test.')

### 1.2 Apple CAPM (daily returns)

In [ ]:
px  = yf.download(['AAPL', '^GSPC'], start='2020-01-01', end='2024-12-31',
                  auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()
ret.columns = ['AAPL', 'SP500']

X_capm = sm.add_constant(ret['SP500'])
capm   = sm.OLS(ret['AAPL'], X_capm).fit()
print(f'n = {len(ret)} trading days, beta_hat = {capm.params["SP500"]:.4f}')

---
# Part 2: Eyes First. The Residual Smile

Plot the residuals of the bond levels model against the fitted values. If the linear form were right, we would see a random cloud. Instead we see the tell-tale SMILE: positive residuals at both ends, negative in the middle. That is curvature the straight line misses.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

o = bond['y10'].argsort()
axes[0].scatter(bond['y10'], bond['P'], s=7, color=GREY, alpha=0.4, label='observed')
axes[0].plot(bond['y10'].iloc[o], m_lvl.fittedvalues.iloc[o], color=RED, lw=2, label='linear fit')
axes[0].set_xlabel('10y Treasury yield (%)'); axes[0].set_ylabel('bond price')
axes[0].set_title('A straight line through a curved world', fontweight='bold', loc='left')
axes[0].legend(frameon=False, fontsize=9)

axes[1].scatter(m_lvl.fittedvalues, m_lvl.resid, s=7, color=RED, alpha=0.4)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xlabel('fitted value'); axes[1].set_ylabel('residual')
axes[1].set_title('The tell-tale smile', fontweight='bold', loc='left')

plt.tight_layout(); plt.show()

---
# Part 3: Ramsey RESET

**The idea:** if the linear model is correct, then nonlinear functions of the fitted values should have nothing left to explain. Add $\hat{y}^2$ and $\hat{y}^3$ to the regression and F-test the two added terms jointly:

$$P_t = \beta_0 + \beta_1 y_t + a_2 \hat{P}_t^{\,2} + a_3 \hat{P}_t^{\,3} + v_t, \qquad H_0: a_2 = a_3 = 0 .$$

### 3.1 By hand

In [ ]:
aux = pd.DataFrame({'y10': bond['y10'],
                    'fit2': m_lvl.fittedvalues**2,
                    'fit3': m_lvl.fittedvalues**3})
m_aux = sm.OLS(bond['P'], sm.add_constant(aux)).fit()

ftest  = m_aux.f_test('fit2 = fit3 = 0')
F_crit = stats.f.ppf(0.95, 2, int(m_aux.df_resid))

print(f'RESET by hand: F = {float(ftest.fvalue):.1f}, p = {float(ftest.pvalue):.2e}')
print(f'q = 2 restrictions, df = {int(m_aux.df_resid)}, F_crit(5%) = {F_crit:.2f}')
print('→ REJECT: the linear form is wrong' if float(ftest.fvalue) > F_crit else '→ do not reject')

### 3.2 One line

In [ ]:
reset = linear_reset(m_lvl, power=3, use_f=True)
print(f'linear_reset: F = {reset.fvalue:.1f}, p = {reset.pvalue:.2e}')
print('Same test, one call. power=3 adds the squared and cubed fitted values.')

---
# Part 4: The Fix. Log the Price, Test Again

Bond prices respond to yields multiplicatively, so model the LOG of the price. Note the exact relationship: $\ln P = \ln 100 - 10 \ln(1 + y/100)$, which is almost linear in $y$ over realistic yield ranges.

In [ ]:
m_log = sm.OLS(np.log(bond['P']), X_b).fit()
reset_log = linear_reset(m_log, power=3, use_f=True)

print(f'Log model:  slope = {m_log.params["y10"]:.4f}')
print(f'RESET on the log model: F = {reset_log.fvalue:.2f}, p = {reset_log.pvalue:.3f}')
print('→ passes' if reset_log.pvalue > 0.05 else '→ still rejects')
print(f'\nFinance payoff: the slope of {m_log.params["y10"]:.4f} per percentage point is')
print('(apart from scaling) minus the bond duration divided by 100. Log coefficients')
print('have their own interpretation language: elasticities. That is the next video.')

*Note: with real yield data the exact F-values will differ from the slides. The point is the contrast: the levels model fails RESET decisively, the log model does not.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].scatter(m_lvl.fittedvalues, m_lvl.resid, s=6, color=RED, alpha=0.4)
axes[0].axhline(0, color='black', lw=1)
axes[0].set_title('Levels model: the smile (RESET rejects)', fontweight='bold', loc='left')
axes[0].set_xlabel('fitted value'); axes[0].set_ylabel('residual')
axes[1].scatter(m_log.fittedvalues, m_log.resid, s=6, color=GREY, alpha=0.4)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Log model: random cloud (RESET passes)', fontweight='bold', loc='left')
axes[1].set_xlabel('fitted value')
plt.tight_layout(); plt.show()

---
# Part 5: Normality. Jarque-Bera on the CAPM Residuals

Jarque-Bera compares two moments of the residuals with their normal reference values, the skewness S (normal: 0) and the kurtosis K (normal: 3):

$$JB = n\left[\frac{S^2}{6} + \frac{(K-3)^2}{24}\right] \sim \chi^2_2 \qquad (crit = 5.99).$$

Both deviations enter as squares, so their sign does not matter and they cannot offset each other.

### 5.1 By hand

In [ ]:
u = capm.resid
S = stats.skew(u)
K = stats.kurtosis(u, fisher=False)     # fisher=False returns the raw kurtosis (normal = 3)
n = len(u)

JB_hand = n * (S**2/6 + (K - 3)**2/24)
print(f'skewness S = {S:.2f}   kurtosis K = {K:.1f}   (normal: 0 / 3)')
print(f'skewness term: {S**2/6:.4f}   kurtosis term: {(K-3)**2/24:.3f}  ← the tails dominate')
print(f'JB = {n} × ({S**2/6:.4f} + {(K-3)**2/24:.3f}) = {JB_hand:.0f}   (χ²(2) crit = 5.99)')
print('→ REJECT normality decisively' if JB_hand > 5.99 else '→ do not reject')

### 5.2 One line, plus the two standard pictures

In [ ]:
jb, jb_p, skew, kurt = jarque_bera(capm.resid)
print(f'jarque_bera: JB = {jb:.0f}, p = {jb_p:.3f}, S = {skew:.2f}, K = {kurt:.1f}')

z = capm.resid / capm.resid.std()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(z, bins=80, density=True, color=YELLOW, edgecolor=GREY, lw=0.3)
xx = np.linspace(-6, 6, 400)
axes[0].plot(xx, stats.norm.pdf(xx), color='black', lw=2, label='normal density')
axes[0].set_xlim(-6, 6); axes[0].legend(frameon=False)
axes[0].set_title('Peaked centre, fat tails', fontweight='bold', loc='left')
axes[0].set_xlabel('standardized residual')
stats.probplot(z, dist='norm', plot=axes[1])
axes[1].set_title('QQ-plot: the tails leave the line', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()
print('Deja vu: fat tails are a stylized fact of returns. Non-normal residuals are the rule.')

**And now the honest question: so what?** With n above one thousand, the central limit theorem keeps the t-tests trustworthy despite the rejection. The result matters for small samples and for everything that depends on the tails, risk measurement in particular. Never assume large-n comfort for monthly strategies with short histories.

---
# Part 6: Spotting Outliers. Standardized Residuals

Divide each residual by the residual standard deviation. Under normality, $|z| > 3$ should occur on about 0.27% of days. Count what we actually observe, and identify the days.

In [ ]:
z = capm.resid / capm.resid.std()
flagged = z[np.abs(z) > 3].sort_values()

expected = 2 * (1 - stats.norm.cdf(3)) * len(z)
print(f'|z| > 3 observed on {len(flagged)} days; normality predicts about {expected:.1f}')
print('\nThe largest five (most negative first):')
print(flagged.head(5).round(2))
print('\nThe largest five positive:')
print(flagged.tail(5).round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(z.index, z, color=GREY, lw=0.5)
for b in (3, -3):
    ax.axhline(b, color=RED, ls=':', lw=1.2)
mask = np.abs(z) > 3
ax.scatter(z.index[mask], z[mask], s=22, color=RED, zorder=3)
ax.set_title('Standardized residuals with the ±3 bands', fontweight='bold', loc='left')
ax.set_ylabel('z')
plt.tight_layout(); plt.show()
print('One z = -5 day carries the squared-residual weight of twenty-five ordinary z = -1 days.')
print('A handful of extreme days can steer the whole estimate. That is why we look at them.')

---
# Part 7: The Fix. Event Dummies, Used Honestly

One 0/1 dummy per identified event day absorbs that day completely: its residual becomes zero, and the remaining coefficients are estimated as if the day were absent, without deleting data. The tool comes straight from the dummy-variables video.

**The honesty rule:** a dummy needs an EVENT you can name, chosen before looking at significance. Dummying away every inconvenient residual is data mining.

In [ ]:
# Choose the three most extreme days, then check they correspond to nameable events
events = z.abs().sort_values(ascending=False).head(3).index
print('Event days used:', [d.date().isoformat() for d in events])

X_d = ret[['SP500']].copy()
for d in events:
    X_d[f'D_{d.date()}'] = (ret.index == d).astype(float)

capm_d = sm.OLS(ret['AAPL'], sm.add_constant(X_d)).fit(cov_type='HC1')
jb_d, jb_d_p, _, _ = jarque_bera(capm_d.resid)
jb0, _, _, _ = jarque_bera(capm.resid)

cmp = pd.DataFrame({
    'beta_hat': [capm.params['SP500'], capm_d.params['SP500']],
    'SE (HC1)': [sm.OLS(ret['AAPL'], X_capm).fit(cov_type='HC1').bse['SP500'], capm_d.bse['SP500']],
    'JB':       [round(jb0), round(jb_d)],
}, index=['no dummies', '3 event dummies']).round(4)
print(cmp)
print('\nReading: the beta barely moves (a REASSURING robustness result), the SE tightens,')
print('and JB falls but still rejects. Fat tails are a property of the whole distribution,')
print('not of three days. Dummies handle events; they do not manufacture normality.')

---
## Summary Table

| Question | Test / Tool | Python | Fix |
|----------|-------------|--------|-----|
| Is the form right? | **Ramsey RESET**: powers of $\hat{y}$, F-test | `linear_reset(model, power=3, use_f=True)` | transform (logs) and re-test |
| Are residuals normal? | **Jarque-Bera**: S and K vs. 0 and 3, $\chi^2_2$ | `jarque_bera(model.resid)` | matters for small n and tails; CLT helps for large n |
| Outliers? | standardized residuals, flag $\|z\| > 3$ | `model.resid / model.resid.std()` | **event dummies**, never silent deletion |

**The hierarchy to remember:** wrong functional form biases the betas (the serious case); non-normality mainly threatens small-sample inference (the mild case); outliers demand a robustness check and an honest treatment.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Logarithmic Transformations & Quadratic Terms.*